In [2]:
!nvidia-smi

Wed Feb 11 15:15:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.95.05              Driver Version: 580.95.05      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5060 ...    Off |   00000000:01:00.0 Off |                  N/A |
| N/A   45C    P8              3W /   35W |     405MiB /   8151MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
import cv2
import random

Select the Points

In [2]:

cap = cv2.VideoCapture("test.mp4")

paused = False
points = []
frame = None

def mouse_callback(event, x, y, flags, param):
    global points, frame, paused
    if paused and event == cv2.EVENT_LBUTTONDOWN:
        points.append((x, y))
        cv2.circle(frame, (x, y), 5, (0, 255, 0), -1)
        cv2.imshow("Video", frame)

cv2.namedWindow("Video")
cv2.setMouseCallback("Video", mouse_callback)

while True:
    if not paused:
        ret, frame = cap.read()
        if not ret:
            break

    cv2.imshow("Video", frame)
    key = cv2.waitKey(30) & 0xFF

    if key == ord("p"):
        paused = not paused

    elif key == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

print("Collected points:")
print(points)


Collected points:
[]


Count Car in Current Frame

Current Number of Cars in a Frame

In [3]:
import cv2
import numpy as np
from ultralytics import YOLO

# Load YOLOv8 model
model = YOLO("runs/detect/train8/weights/best.pt")

# ROI points (4 points per ROI)
points = [
    (1273, 550), (822, 313), (756, 323), (1183, 686),
    (1008, 706), (1179, 682), (762, 323), (719, 329),
    (791, 713), (1002, 702), (723, 332), (673, 332),
]

# Group points into ROIs (4 points per ROI)
rois = []
for i in range(0, len(points), 4):
    roi = points[i:i+4]
    if len(roi) == 4:
        rois.append(np.array(roi, dtype=np.int32))

# Colors for ROIs
roi_colors = [
    (255, 0, 0), (0, 255, 0), (0, 0, 255), (255, 255, 0),
    (255, 0, 255), (0, 255, 255)
]

# Open video
cap = cv2.VideoCapture("test.mp4")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    overlay = frame.copy()

    # Draw ROIs
    for idx, roi in enumerate(rois):
        cv2.fillPoly(overlay, [roi], roi_colors[idx % len(roi_colors)])
        cv2.polylines(frame, [roi], True, roi_colors[idx % len(roi_colors)], 2)

    # Run YOLOv8 detection
    results = model.predict(frame)
    detections = results[0].boxes.data.cpu().numpy()  # x1, y1, x2, y2, score, class

    # Initialize counts per ROI
    roi_counts = [0] * len(rois)

    for det in detections:
        x1, y1, x2, y2, score, cls = det
        x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
        cx, cy = (x1 + x2)//2, (y1 + y2)//2

        # Check which ROI contains the bbox center
        for idx, roi in enumerate(rois):
            if cv2.pointPolygonTest(roi, (cx, cy), False) >= 0:
                # Draw bounding box only if inside ROI
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 255), 2)
                roi_counts[idx] += 1
                break  # No need to check other ROIs

    # Blend overlay for ROI transparency
    cv2.addWeighted(overlay, 0.3, frame, 0.7, 0, frame)

    # Show counts per ROI
    for idx, count in enumerate(roi_counts):
        cv2.putText(frame, f"Lane {idx+1}: {count} cars", (10, 30 + idx*30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, roi_colors[idx % len(roi_colors)], 2)

    cv2.imshow("Cars Inside ROI with Count", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()



0: 288x512 2 Trucks, 9 Cars, 1 Bus, 904.4ms
Speed: 10.1ms preprocess, 904.4ms inference, 12.6ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 2 Trucks, 9 Cars, 1 Bus, 587.1ms
Speed: 3.7ms preprocess, 587.1ms inference, 6.1ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 2 Trucks, 10 Cars, 1 Bus, 865.2ms
Speed: 8.5ms preprocess, 865.2ms inference, 15.8ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 2 Trucks, 9 Cars, 1 Van, 1 Bus, 866.8ms
Speed: 5.4ms preprocess, 866.8ms inference, 22.9ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 2 Trucks, 10 Cars, 1 Van, 1 Bus, 835.0ms
Speed: 7.7ms preprocess, 835.0ms inference, 21.4ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 2 Trucks, 10 Cars, 1 Bus, 580.5ms
Speed: 6.1ms preprocess, 580.5ms inference, 4.3ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 2 Trucks, 10 Cars, 1 Bus, 682.7ms
Speed: 3.8ms preprocess, 682.7ms inference, 22.7ms postprocess per image a

In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np

# -------------------------------
# 1️⃣ Load YOLOv11 model
# -------------------------------
model = YOLO("runs/detect/train8/weights/best.pt")

# -------------------------------
# 2️⃣ Define ROI polygon
# -------------------------------
roi_points = np.array([[1273, 550],
                       [822, 313],
                       [756, 323],
                       [1183, 686]], np.int32)
roi_points = roi_points.reshape((-1, 1, 2))

# -------------------------------
# 3️⃣ Initialize tracking
# -------------------------------
# YOLO built-in tracker (BYTETRACK)
tracker_type = "bytetrack.yaml"  # comes with Ultralytics
tracked_ids = set()  # store unique IDs counted
total_count = 0

# -------------------------------
# 4️⃣ Process video
# -------------------------------
cap = cv2.VideoCapture("test.mp4")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # -------------------------------
    # Run detection + tracking
    # -------------------------------
    results = model.track(source=frame, tracker=tracker_type, persist=True)[0]

    # -------------------------------
    # Draw ROI
    # -------------------------------
    cv2.polylines(frame, [roi_points], True, (0, 255, 0), 2)

    # -------------------------------
    # Filter tracked objects inside ROI
    # -------------------------------
    for obj in results.boxes.data.tolist():  # [x1, y1, x2, y2, score, class, id]
        x1, y1, x2, y2, conf, cls, obj_id = obj
        obj_id = int(obj_id)
        cx = int((x1 + x2) / 2)
        cy = int((y1 + y2) / 2)

        # Only count objects inside ROI
        if cv2.pointPolygonTest(roi_points, (cx, cy), False) >= 0:
            # Count unique IDs
            if obj_id not in tracked_ids:
                total_count += 1
                tracked_ids.add(obj_id)

            # Draw bounding box and ID
            cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 0, 255), 2)
            cv2.putText(frame, f"ID: {obj_id}", (int(x1), int(y1)-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 2)

    # -------------------------------
    # Display total car count
    # -------------------------------
    cv2.putText(frame, f"Total Cars: {total_count}", (50, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

    cv2.imshow("Frame", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


single lane testing

In [3]:
from ultralytics import YOLO
import cv2
import numpy as np
from scipy.spatial import distance

# -------------------------------
# Load model
# -------------------------------
model = YOLO("runs/detect/train8/weights/best.pt")

# -------------------------------
# Define ROI
# -------------------------------

roi_points = np.array([[1162, 671],
                       [1271, 563],
                       [963, 391],
                       [866, 418]], np.int32)
roi_points = roi_points.reshape((-1, 1, 2))

# -------------------------------
# Centroid tracking
# -------------------------------
next_object_id = 1
objects = {}  # object_id: (cx, cy)

max_distance = 100  # maximum distance to match centroids

total_count = 0

# -------------------------------
# Open video
# -------------------------------
cap = cv2.VideoCapture("test.mp4")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    overlay = frame.copy()

    results = model(frame)[0]  # detect objects

    current_centroids = []

    for box in results.boxes.data.tolist():  # [x1, y1, x2, y2, conf, cls]
        x1, y1, x2, y2, conf, cls = box
        cx = int((x1 + x2) / 2)
        cy = int((y1 + y2) / 2)
    
        if cv2.pointPolygonTest(roi_points, (cx, cy), False) >= 0:
            current_centroids.append((cx, cy, x1, y1, x2, y2))

    # -------------------------------
    # Match centroids to existing objects
    # -------------------------------
    new_objects = {}

    for cx, cy, x1, y1, x2, y2 in current_centroids:
        matched = False
        for object_id, (ox, oy) in objects.items():
            if distance.euclidean((cx, cy), (ox, oy)) < max_distance:
                new_objects[object_id] = (cx, cy)
                matched = True
                break

        if not matched:
            new_objects[next_object_id] = (cx, cy)
            total_count += 1
            next_object_id += 1

    objects = new_objects

    # -------------------------------
    # Draw boxes and IDs
    # -------------------------------
    for object_id, (cx, cy) in objects.items():
        cv2.circle(frame, (cx, cy), 5, (0, 0, 255), -1)
        cv2.putText(frame, f"ID: {object_id}", (cx-10, cy-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 2)

    # Draw ROI
    # Draw ROI with transparency
    cv2.fillPoly(overlay, [roi_points], (0, 255, 0))
    
    alpha = 0.3
    frame = cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0)
    
    # Draw ROI border on top
    cv2.polylines(frame, [roi_points], True, (0, 255, 0), 2)


    # Show total
    cv2.putText(frame, f"Total: {total_count}", (20, 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    cv2.imshow("Frame", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()



0: 288x512 2 Trucks, 10 Cars, 1 Van, 1 Bus, 8.0ms
Speed: 1.2ms preprocess, 8.0ms inference, 2.9ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 2 Trucks, 10 Cars, 1 Van, 1 Bus, 8.3ms
Speed: 1.2ms preprocess, 8.3ms inference, 3.2ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 2 Trucks, 10 Cars, 1 Van, 1 Bus, 12.0ms
Speed: 2.1ms preprocess, 12.0ms inference, 3.1ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 2 Trucks, 10 Cars, 2 Vans, 1 Bus, 8.2ms
Speed: 4.7ms preprocess, 8.2ms inference, 3.1ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 2 Trucks, 10 Cars, 2 Vans, 1 Bus, 8.4ms
Speed: 1.1ms preprocess, 8.4ms inference, 3.2ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 2 Trucks, 10 Cars, 1 Van, 1 Bus, 8.5ms
Speed: 1.3ms preprocess, 8.5ms inference, 4.1ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 2 Trucks, 10 Cars, 1 Van, 1 Bus, 8.4ms
Speed: 1.3ms preprocess, 8.4ms inference, 3.6ms postprocess per

Total Car Count

In [2]:
from ultralytics import YOLO
import cv2
import numpy as np
from scipy.spatial import distance

# -------------------------------
# Load YOLO model
# -------------------------------
model = YOLO("runs/detect/train8/weights/best.pt")

# -------------------------------
# ROI points (each 4 points = one ROI)
# -------------------------------
points = [(786, 689), (983, 680), (814, 459), (714, 471), 
          (984, 680), (1165, 663), (907, 450), (815, 458), 
          (1167, 663), (1275, 589), (1053, 434), (909, 450)]

# -------------------------------
# ROI colors
# -------------------------------
roi_colors = [
    (255, 0, 0),    # Blue
    (0, 255, 0),    # Green
    (0, 0, 255),    # Red
    (255, 255, 0),  # Cyan
    (255, 0, 255),  # Magenta
    (0, 255, 255)   # Yellow
]

# -------------------------------
# Prepare ROI structures
# -------------------------------
roi_polygons = []
roi_contours = []
roi_counts = []
roi_objects = []

for i in range(0, len(points), 4):
    roi = points[i:i+4]
    if len(roi) == 4:
        poly = np.array(roi, dtype=np.int32)
        roi_polygons.append(poly)
        roi_contours.append(poly.reshape((-1, 1, 2)))
        roi_counts.append(0)
        roi_objects.append({})  # object_id -> (cx, cy)

# -------------------------------
# Tracking parameters
# -------------------------------
next_object_id = 1
max_distance = 100

# -------------------------------
# Open video
# -------------------------------
cap = cv2.VideoCapture("test.mp4")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    overlay = frame.copy()
    results = model(frame)[0]

    detections = []

    # -------------------------------
    # Detect objects
    # -------------------------------
    for box in results.boxes.data.tolist():
        x1, y1, x2, y2, conf, cls = box
        cx = int((x1 + x2) / 2)
        cy = int((y1 + y2) / 2)
        detections.append((cx, cy))

    # -------------------------------
    # Process each ROI independently
    # -------------------------------
    for idx, poly in enumerate(roi_polygons):
        new_objects = {}

        for cx, cy in detections:
            if cv2.pointPolygonTest(poly, (cx, cy), False) >= 0:
                matched = False
                for obj_id, (ox, oy) in roi_objects[idx].items():
                    if distance.euclidean((cx, cy), (ox, oy)) < max_distance:
                        new_objects[obj_id] = (cx, cy)
                        matched = True
                        break
                if not matched:
                    new_objects[next_object_id] = (cx, cy)
                    roi_counts[idx] += 1
                    next_object_id += 1

        roi_objects[idx] = new_objects

        # Draw tracked centroids
        for cx, cy in roi_objects[idx].values():
            cv2.circle(frame, (cx, cy), 4, roi_colors[idx % len(roi_colors)], -1)

    # -------------------------------
    # Draw transparent ROIs
    # -------------------------------
    for i, contour in enumerate(roi_contours):
        color = roi_colors[i % len(roi_colors)]
        cv2.fillPoly(overlay, [contour], color)
        cv2.polylines(frame, [contour], True, color, 2)

    frame = cv2.addWeighted(overlay, 0.3, frame, 0.7, 0)  # Transparent overlay

    # -------------------------------
    # Draw per-lane counts in top-left corner with transparency
    # -------------------------------
    start_x, start_y = 20, 30
    line_height = 25

    # Transparent background rectangle
    overlay_counts = frame.copy()
    rect_height = len(roi_counts) * line_height + 10
    cv2.rectangle(overlay_counts, (10, 5), (220, 5 + rect_height), (0, 0, 0), -1)
    frame = cv2.addWeighted(overlay_counts, 0.5, frame, 0.5, 0)

    # Draw text for each ROI
    for idx, count in enumerate(roi_counts):
        color = roi_colors[idx % len(roi_colors)]
        cv2.putText(
            frame,
            f"ROI {idx + 1}: {count}",
            (start_x, start_y + idx * line_height),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            color,
            2
        )

    cv2.imshow("Frame", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()



0: 288x512 2 Trucks, 10 Cars, 1 Van, 1 Bus, 7.5ms
Speed: 1.1ms preprocess, 7.5ms inference, 2.8ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 2 Trucks, 10 Cars, 1 Van, 1 Bus, 7.4ms
Speed: 1.2ms preprocess, 7.4ms inference, 3.0ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 2 Trucks, 10 Cars, 1 Van, 1 Bus, 8.7ms
Speed: 1.3ms preprocess, 8.7ms inference, 3.1ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 2 Trucks, 10 Cars, 2 Vans, 1 Bus, 8.0ms
Speed: 1.3ms preprocess, 8.0ms inference, 3.2ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 2 Trucks, 10 Cars, 2 Vans, 1 Bus, 7.6ms
Speed: 1.3ms preprocess, 7.6ms inference, 3.2ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 2 Trucks, 10 Cars, 1 Van, 1 Bus, 7.7ms
Speed: 1.2ms preprocess, 7.7ms inference, 3.1ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 2 Trucks, 10 Cars, 1 Van, 1 Bus, 7.8ms
Speed: 1.4ms preprocess, 7.8ms inference, 3.2ms postprocess per i

In [3]:
from ultralytics import YOLO
import cv2
import numpy as np
from scipy.spatial import distance

# -------------------------------
# Load YOLO model
# -------------------------------
model = YOLO("runs/detect/train8/weights/best.pt")

# -------------------------------
# ROI points (each 4 points = one ROI)
# -------------------------------
points = [
    (834, 392), (932, 370), (1247, 538), (1101, 619),
    (1098, 611), (971, 663), (787, 420), (847, 406),
    (786, 698), (972, 663), (789, 423), (704, 434),
    (505, 698), (568, 454), (467, 448), (295, 672),
    (295, 672), (122, 657), (360, 457), (466, 448),
    (13, 605), (123, 658), (360, 459), (260, 452)
]

# -------------------------------
# ROI colors
# -------------------------------
roi_colors = [
    (255, 0, 0),    # Blue
    (0, 255, 0),    # Green
    (0, 0, 255),    # Red
    (255, 255, 0),  # Cyan
    (255, 0, 255),  # Magenta
    (0, 255, 255)   # Yellow
]

# -------------------------------
# Prepare ROI structures
# -------------------------------
roi_polygons = []
roi_contours = []
roi_counts = []
roi_objects = []

for i in range(0, len(points), 4):
    roi = points[i:i+4]
    if len(roi) == 4:
        poly = np.array(roi, dtype=np.int32)
        roi_polygons.append(poly)
        roi_contours.append(poly.reshape((-1, 1, 2)))
        roi_counts.append(0)
        roi_objects.append({})  # object_id -> (cx, cy, x1, y1, x2, y2)

# -------------------------------
# Tracking parameters
# -------------------------------
next_object_id = 1
max_distance = 100

# -------------------------------
# Open video
# -------------------------------
cap = cv2.VideoCapture("test.mp4")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    overlay = frame.copy()
    results = model(frame)[0]

    # -------------------------------
    # Process each ROI independently
    # -------------------------------
    for idx, poly in enumerate(roi_polygons):
        new_objects = {}

        for box in results.boxes.data.tolist():
            x1, y1, x2, y2, conf, cls = box
            cx = int((x1 + x2) / 2)
            cy = int((y1 + y2) / 2)

            # Select reference point based on ROI index
            if idx < 3:  # first 3 ROIs: bottom-right
                ref_point = (int(x2), int(y2))
            else:        # remaining ROIs: bottom-left
                ref_point = (int(x1), int(y2))

            # Count if reference point is inside ROI
            if cv2.pointPolygonTest(poly, ref_point, False) >= 0:
                matched = False
                for obj_id, (ox, oy, ox1, oy1, ox2, oy2) in roi_objects[idx].items():
                    if distance.euclidean((cx, cy), (ox, oy)) < max_distance:
                        new_objects[obj_id] = (cx, cy, x1, y1, x2, y2)
                        matched = True
                        break

                if not matched:
                    new_objects[next_object_id] = (cx, cy, x1, y1, x2, y2)
                    roi_counts[idx] += 1
                    next_object_id += 1

        roi_objects[idx] = new_objects

        # Draw tracked objects with ID at bottom-left of bounding box
        for obj_id, (cx, cy, x1, y1, x2, y2) in roi_objects[idx].items():
            cv2.circle(frame, (cx, cy), 4, roi_colors[idx % len(roi_colors)], -1)
            # Draw ID near bottom-left of bbox (for visibility)
            cv2.putText(frame, f"ID: {obj_id}", (int(x1), int(y2)+15),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, roi_colors[idx % len(roi_colors)], 2)

    # -------------------------------
    # Draw transparent ROIs
    # -------------------------------
    for i, contour in enumerate(roi_contours):
        color = roi_colors[i % len(roi_colors)]
        cv2.fillPoly(overlay, [contour], color)
        cv2.polylines(frame, [contour], True, color, 2)

    frame = cv2.addWeighted(overlay, 0.3, frame, 0.7, 0)  # Transparent overlay

    # -------------------------------
    # Draw per-lane counts in top-left corner with transparency
    # -------------------------------
    start_x, start_y = 20, 30
    line_height = 25
    overlay_counts = frame.copy()
    rect_height = len(roi_counts) * line_height + 10
    cv2.rectangle(overlay_counts, (10, 5), (220, 5 + rect_height), (0, 0, 0), -1)
    frame = cv2.addWeighted(overlay_counts, 0.5, frame, 0.5, 0)

    for idx, count in enumerate(roi_counts):
        color = roi_colors[idx % len(roi_colors)]
        cv2.putText(frame, f"ROI {idx + 1}: {count}",
                    (start_x, start_y + idx * line_height),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

    cv2.imshow("Frame", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()



0: 288x512 2 Trucks, 10 Cars, 1 Van, 1 Bus, 8.5ms
Speed: 1.0ms preprocess, 8.5ms inference, 4.4ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 2 Trucks, 10 Cars, 1 Van, 1 Bus, 13.8ms
Speed: 2.8ms preprocess, 13.8ms inference, 3.2ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 2 Trucks, 10 Cars, 1 Van, 1 Bus, 8.3ms
Speed: 1.3ms preprocess, 8.3ms inference, 3.1ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 2 Trucks, 10 Cars, 2 Vans, 1 Bus, 8.1ms
Speed: 1.3ms preprocess, 8.1ms inference, 3.3ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 2 Trucks, 10 Cars, 2 Vans, 1 Bus, 7.9ms
Speed: 1.3ms preprocess, 7.9ms inference, 3.3ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 2 Trucks, 10 Cars, 1 Van, 1 Bus, 9.6ms
Speed: 1.6ms preprocess, 9.6ms inference, 3.8ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 2 Trucks, 10 Cars, 1 Van, 1 Bus, 8.6ms
Speed: 3.4ms preprocess, 8.6ms inference, 3.4ms postprocess per

In [ ]:
save video 

In [1]:
import cv2
import numpy as np
from ultralytics import YOLO

# -------------------------------
# Load YOLOv8 model
# -------------------------------
model = YOLO("runs/detect/train8/weights/best.pt")

# -------------------------------
# ROI points (4 points per ROI)
# -------------------------------
points = [
    (1273, 550), (822, 313), (756, 323), (1183, 686),
    (1008, 706), (1179, 682), (762, 323), (719, 329),
    (791, 713), (1002, 702), (723, 332), (673, 332),
]

# Group points into ROIs (4 points per ROI)
rois = []
for i in range(0, len(points), 4):
    roi = points[i:i+4]
    if len(roi) == 4:
        rois.append(np.array(roi, dtype=np.int32))

# Colors for ROIs
roi_colors = [
    (255, 0, 0), (0, 255, 0), (0, 0, 255),
    (255, 255, 0), (255, 0, 255), (0, 255, 255)
]

# -------------------------------
# Open video
# -------------------------------
cap = cv2.VideoCapture("test.mp4")

# -------------------------------
# Video writer to save output
# -------------------------------
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
if fps == 0:
    fps = 30  # fallback

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter("output_roi_count.mp4", fourcc, fps, (width, height))

# -------------------------------
# Process video
# -------------------------------
while True:
    ret, frame = cap.read()
    if not ret:
        break

    overlay = frame.copy()

    # Draw ROIs
    for idx, roi in enumerate(rois):
        cv2.fillPoly(overlay, [roi], roi_colors[idx % len(roi_colors)])
        cv2.polylines(frame, [roi], True, roi_colors[idx % len(roi_colors)], 2)

    # Run YOLOv8 detection
    results = model.predict(frame)
    detections = results[0].boxes.data.cpu().numpy()  # x1, y1, x2, y2, score, class

    # Initialize counts per ROI
    roi_counts = [0] * len(rois)

    for det in detections:
        x1, y1, x2, y2, score, cls = det
        x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
        cx, cy = (x1 + x2) // 2, (y1 + y2) // 2

        # Check which ROI contains the bbox center
        for idx, roi in enumerate(rois):
            if cv2.pointPolygonTest(roi, (cx, cy), False) >= 0:
                # Draw bounding box only if inside ROI
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 255), 2)
                roi_counts[idx] += 1
                break  # No need to check other ROIs

    # Blend overlay for ROI transparency
    cv2.addWeighted(overlay, 0.3, frame, 0.7, 0, frame)

    # Show counts per ROI
    for idx, count in enumerate(roi_counts):
        cv2.putText(frame, f"Lane {idx+1}: {count} cars", (10, 30 + idx * 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, roi_colors[idx % len(roi_colors)], 2)

    # Show and save frame
    cv2.imshow("Cars Inside ROI with Count", frame)
    out.write(frame)  # <<< SAVE VIDEO HERE

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# -------------------------------
# Cleanup
# -------------------------------
cap.release()
out.release()
cv2.destroyAllWindows()


AcceleratorError: CUDA error: CUDA-capable device(s) is/are busy or unavailable
Search for `cudaErrorDevicesUnavailable' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [4]:
from ultralytics import YOLO
import cv2
import numpy as np
from scipy.spatial import distance

# -------------------------------
# Load YOLO model
# -------------------------------
model = YOLO("runs/detect/train8/weights/best.pt")

# -------------------------------
# ROI points (each 4 points = one ROI)
# -------------------------------
points = [
    (786, 689), (983, 680), (814, 459), (714, 471),
    (984, 680), (1165, 663), (907, 450), (815, 458),
    (1167, 663), (1275, 589), (1053, 434), (909, 450)
]

# -------------------------------
# ROI colors
# -------------------------------
roi_colors = [
    (255, 0, 0),    # Blue
    (0, 255, 0),    # Green
    (0, 0, 255),    # Red
    (255, 255, 0),  # Cyan
    (255, 0, 255),  # Magenta
    (0, 255, 255)   # Yellow
]

# -------------------------------
# Prepare ROI structures
# -------------------------------
roi_polygons = []
roi_contours = []
total_counts = []   # cumulative count
roi_objects = []    # tracking objects

for i in range(0, len(points), 4):
    roi = points[i:i+4]
    if len(roi) == 4:
        poly = np.array(roi, dtype=np.int32)
        roi_polygons.append(poly)
        roi_contours.append(poly.reshape((-1, 1, 2)))
        total_counts.append(0)
        roi_objects.append({})  # object_id -> (cx, cy)

# -------------------------------
# Tracking parameters
# -------------------------------
next_object_id = 1
max_distance = 100

# -------------------------------
# Open video
# -------------------------------
cap = cv2.VideoCapture("test.mp4")

# Get video properties for saving
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

# Define video writer
out = cv2.VideoWriter(
    'output_with_counts.mp4',
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps,
    (frame_width, frame_height)
)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    overlay = frame.copy()
    results = model(frame)[0]

    detections = []

    # -------------------------------
    # Detect objects and get centroids
    # -------------------------------
    for box in results.boxes.data.tolist():
        x1, y1, x2, y2, conf, cls = box
        cx = int((x1 + x2) / 2)
        cy = int((y1 + y2) / 2)
        detections.append((cx, cy))

    # -------------------------------
    # Count total and per-frame cars per ROI
    # -------------------------------
    per_frame_counts = [0] * len(roi_polygons)

    for idx, poly in enumerate(roi_polygons):
        new_objects = {}

        for cx, cy in detections:
            if cv2.pointPolygonTest(poly, (cx, cy), False) >= 0:
                # Total cars tracking
                matched = False
                for obj_id, (ox, oy) in roi_objects[idx].items():
                    if distance.euclidean((cx, cy), (ox, oy)) < max_distance:
                        new_objects[obj_id] = (cx, cy)
                        matched = True
                        break
                if not matched:
                    new_objects[next_object_id] = (cx, cy)
                    total_counts[idx] += 1
                    next_object_id += 1

                # Per-frame count
                per_frame_counts[idx] += 1

        roi_objects[idx] = new_objects

        # Draw tracked centroids
        for cx, cy in roi_objects[idx].values():
            cv2.circle(frame, (cx, cy), 4, roi_colors[idx % len(roi_colors)], -1)

    # -------------------------------
    # Draw transparent ROIs
    # -------------------------------
    for i, contour in enumerate(roi_contours):
        color = roi_colors[i % len(roi_colors)]
        cv2.fillPoly(overlay, [contour], color)
        cv2.polylines(frame, [contour], True, color, 2)

    frame = cv2.addWeighted(overlay, 0.3, frame, 0.7, 0)

    # -------------------------------
    # Draw counts on left and right side
    # -------------------------------
    start_y = 30
    line_height = 25
    left_x = 10            # Left side for total counts
    right_x = frame.shape[1] - 220  # Right side for per-frame counts

    # Transparent background for left side
    overlay_left = frame.copy()
    rect_height = len(total_counts) * line_height + 10
    cv2.rectangle(overlay_left, (5, 5), (220, 5 + rect_height), (0, 0, 0), -1)
    frame = cv2.addWeighted(overlay_left, 0.5, frame, 0.5, 0)

    # Transparent background for right side
    overlay_right = frame.copy()
    cv2.rectangle(overlay_right, (right_x - 10, 5), (right_x + 210, 5 + rect_height), (0, 0, 0), -1)
    frame = cv2.addWeighted(overlay_right, 0.5, frame, 0.5, 0)

    for idx in range(len(total_counts)):
        color = roi_colors[idx % len(roi_colors)]
        # Left: Total cars
        cv2.putText(frame, f"Total ROI {idx+1}: {total_counts[idx]}",
                    (left_x, start_y + idx * line_height),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
        # Right: Cars per frame
        cv2.putText(frame, f"Frame ROI {idx+1}: {per_frame_counts[idx]}",
                    (right_x, start_y + idx * line_height),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

    # Write frame to video
    out.write(frame)

    # Show frame
    cv2.imshow("Combined Car Counts", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
out.release()
cv2.destroyAllWindows()
print("Video saved as output_with_counts.mp4")



0: 288x512 2 Trucks, 10 Cars, 1 Van, 1 Bus, 7.5ms
Speed: 1.0ms preprocess, 7.5ms inference, 2.9ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 2 Trucks, 10 Cars, 1 Van, 1 Bus, 13.3ms
Speed: 1.4ms preprocess, 13.3ms inference, 3.4ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 2 Trucks, 10 Cars, 1 Van, 1 Bus, 8.5ms
Speed: 1.5ms preprocess, 8.5ms inference, 3.7ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 2 Trucks, 10 Cars, 2 Vans, 1 Bus, 7.5ms
Speed: 3.3ms preprocess, 7.5ms inference, 3.2ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 2 Trucks, 10 Cars, 2 Vans, 1 Bus, 8.6ms
Speed: 1.8ms preprocess, 8.6ms inference, 3.3ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 2 Trucks, 10 Cars, 1 Van, 1 Bus, 8.1ms
Speed: 1.3ms preprocess, 8.1ms inference, 3.2ms postprocess per image at shape (1, 3, 288, 512)

0: 288x512 2 Trucks, 10 Cars, 1 Van, 1 Bus, 7.6ms
Speed: 1.2ms preprocess, 7.6ms inference, 3.1ms postprocess per